<h1>SQLAlchemy Tutorial<h1/>

In [1]:
import sqlalchemy

In [2]:
sqlalchemy.__version__

'2.0.46'

In [3]:
from sqlalchemy import create_engine, text

In [4]:
engine = create_engine("sqlite+pysqlite:///:memory:", echo=True)

In [5]:
with engine.connect() as conn:
    result =  conn.execute(text("select 'hello world'"))
    print(result.all())

2026-01-24 14:04:27,533 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:27,535 INFO sqlalchemy.engine.Engine select 'hello world'
2026-01-24 14:04:27,537 INFO sqlalchemy.engine.Engine [generated in 0.00430s] ()
[('hello world',)]
2026-01-24 14:04:27,539 INFO sqlalchemy.engine.Engine ROLLBACK


In [6]:
# Commit as you go"
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE some_table (x int, y int)"))
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"), 
        [{"x": 1, "y": 1}, {"x": 2, "y": 4}],
    )
    conn.commit()

2026-01-24 14:04:27,556 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:27,557 INFO sqlalchemy.engine.Engine CREATE TABLE some_table (x int, y int)
2026-01-24 14:04:27,559 INFO sqlalchemy.engine.Engine [generated in 0.00326s] ()
2026-01-24 14:04:27,564 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-24 14:04:27,566 INFO sqlalchemy.engine.Engine [generated in 0.00231s] [(1, 1), (2, 4)]
2026-01-24 14:04:27,568 INFO sqlalchemy.engine.Engine COMMIT


In [7]:
# begins once#
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 6, "y": 8}, {"x": 9, "y": 10}],
    )

2026-01-24 14:04:27,587 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:27,588 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-24 14:04:27,590 INFO sqlalchemy.engine.Engine [cached since 0.02605s ago] [(6, 8), (9, 10)]
2026-01-24 14:04:27,590 INFO sqlalchemy.engine.Engine COMMIT


In [8]:
# begins once#
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (9, 10)")
            )

2026-01-24 14:04:27,617 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:27,619 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (9, 10)
2026-01-24 14:04:27,620 INFO sqlalchemy.engine.Engine [generated in 0.00115s] ()
2026-01-24 14:04:27,621 INFO sqlalchemy.engine.Engine COMMIT


In [9]:
# select statement#
with engine.begin() as conn:
    query_result = conn.execute(text("SELECT * FROM some_table"))
    print(query_result.all())

2026-01-24 14:04:27,648 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:27,649 INFO sqlalchemy.engine.Engine SELECT * FROM some_table
2026-01-24 14:04:27,650 INFO sqlalchemy.engine.Engine [generated in 0.00089s] ()
[(1, 1), (2, 4), (6, 8), (9, 10), (9, 10)]
2026-01-24 14:04:27,652 INFO sqlalchemy.engine.Engine COMMIT


In [10]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table"))
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-24 14:04:27,728 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:27,730 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table
2026-01-24 14:04:27,732 INFO sqlalchemy.engine.Engine [generated in 0.05430s] ()
x: 1 y: 1
x: 2 y: 4
x: 6 y: 8
x: 9 y: 10
x: 9 y: 10
2026-01-24 14:04:27,735 INFO sqlalchemy.engine.Engine ROLLBACK


<h2>Sending Parameters<h2/>

In [11]:
# return value of y where its value is greater than a specific value)

with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table WHERE y > :y"), {"y": 8})
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-24 14:04:27,757 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:27,759 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ?
2026-01-24 14:04:27,760 INFO sqlalchemy.engine.Engine [generated in 0.00215s] (8,)
x: 9 y: 10
x: 9 y: 10
2026-01-24 14:04:27,761 INFO sqlalchemy.engine.Engine ROLLBACK


<h2>Sending Multiple Parameters<h2/>

In [12]:
# inserting multiple records in a sql statement

with engine.connect() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"), 
        [{"x": 11, "y": 12}, {"x": 13, "y": 14}]
    )
    conn.commit()

2026-01-24 14:04:27,786 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:27,787 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-24 14:04:27,789 INFO sqlalchemy.engine.Engine [cached since 0.2248s ago] [(11, 12), (13, 14)]
2026-01-24 14:04:27,790 INFO sqlalchemy.engine.Engine COMMIT


<h2>Executing with an ORM Session<h2/>

In [13]:
from sqlalchemy.orm import Session

In [14]:
stmt = text("SELECT x, y FROM some_table WHERE y > :y ORDER BY x, y")
with Session(engine) as session:
    result = session.execute(stmt, {"y": 6})
    for row in result:
        print(f"x: {row.x}, y: {row.y}")

2026-01-24 14:04:27,929 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:27,931 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ? ORDER BY x, y
2026-01-24 14:04:27,932 INFO sqlalchemy.engine.Engine [generated in 0.00143s] (6,)
x: 6, y: 8
x: 9, y: 10
x: 9, y: 10
x: 11, y: 12
x: 13, y: 14
2026-01-24 14:04:27,934 INFO sqlalchemy.engine.Engine ROLLBACK


In [15]:
# commit #

with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11},{"x": 13, "y": 15}],
    )
    session.commit()

2026-01-24 14:04:27,956 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:27,958 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-24 14:04:27,959 INFO sqlalchemy.engine.Engine [generated in 0.00089s] [(11, 9), (15, 13)]
2026-01-24 14:04:27,960 INFO sqlalchemy.engine.Engine COMMIT


In [16]:
with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11}, {"x": 13, "y": 15}]
    )
    session.commit()

2026-01-24 14:04:27,973 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:27,975 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-24 14:04:27,976 INFO sqlalchemy.engine.Engine [cached since 0.01826s ago] [(11, 9), (15, 13)]
2026-01-24 14:04:27,978 INFO sqlalchemy.engine.Engine COMMIT


<h2>Setting up MetaData with Table objects<h2/>

In [17]:
from sqlalchemy import MetaData
metadata_obj = MetaData()

In [18]:
from sqlalchemy import Table, Column, Integer, String
user_table = Table(
    "user_account",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("name", String(30)),
    Column("fullname", String),
)

In [19]:
user_table.c.name

Column('name', String(length=30), table=<user_account>)

In [20]:
user_table.c.keys()

['id', 'name', 'fullname']

In [21]:
user_table.primary_key

PrimaryKeyConstraint(Column('id', Integer(), table=<user_account>, primary_key=True, nullable=False))

In [22]:
# create a second table
from sqlalchemy import ForeignKey

address_table = Table(
    "address",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("user_id", ForeignKey("user_account.id"), nullable=False),
    Column("email_address", String, nullable=False)
)

In [23]:
metadata_obj.create_all(engine)

2026-01-24 14:04:28,116 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 14:04:28,118 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-01-24 14:04:28,120 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-24 14:04:28,121 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2026-01-24 14:04:28,122 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-24 14:04:28,124 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2026-01-24 14:04:28,126 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-24 14:04:28,127 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("address")
2026-01-24 14:04:28,129 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-24 14:04:28,132 INFO sqlalchemy.engine.Engine 
CREATE TABLE user_account (
	id INTEGER NOT NULL, 
	name VARCHAR(30), 
	fullname VARCHAR, 
	PRIMARY KEY (id)
)


2026-01-24 14:04:28,133 INFO sqlalchemy.engine.Engine [no key 0.00131s] ()
2026-01-24 14:04:28,136 INFO sqlalchemy.engine.Engine 
C

<h2>Establishing a Declarative Base<h2/>

In [25]:
# create a new class that subclasses the SQLAlchemy DeclarativeBase class

from sqlalchemy.orm import DeclarativeBase

class Base(DeclarativeBase):
    pass

In [26]:
Base.metadata

MetaData()

In [27]:
Base.registry

In [ ]:
from typing import List
from typing import Optional
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

class User(Base):
    __tablename__ = "user_account"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[Optional[str]]

    addresses: Mapped[List["Address"]] = relationship(back_populates="user")

    def __repr__(self) -> str:
        return f"User(id={self.id!r}, fullname={self.fullname!r})"
    

class Address(Base):
    __tablename__ = "address"

    id: Mapped[int] = mapped_column(primary_key=True)
    email_address: Mapped[str]
    user_id = mapped_column(ForeignKey("user_account.id"))

    user: Mapped[User] = relationship(back_populates="addresses")

    def __repr__(self) -> str:
        return f"Address(id=)"
        


